# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YanoLeechiu/FlyRankAIProjects/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring (provisional).**

In Week 1 I found that search volume barely tracks the traffic a page actually gets. That means the obvious rule, "refresh the pages with the biggest keywords first", points editors at the wrong pages. Meanwhile more than half the pages in the starter data are trending down, far more than any team can review. So the useful problem is not "which pages are declining?" but "which declining or at-risk pages should a person look at first?" That is a ranking problem, and it is what this lane is about.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Search question:** Among a client's existing pages, which ones should be reviewed first for a refresh, based only on signals observed before the decision?

- **Unit of analysis:** one page (content item), scored at a single decision point.
- **Output:** a ranked review queue, where each page gets a score, a suggested action (refresh, expand, protect, monitor) and reason codes that say why.
- **Decision it improves:** where a content editor spends a limited weekly review budget (assume about 50 pages).
- **Who acts and how:** the editor opens the top of the queue, checks the reason codes, and decides whether to update, expand, or leave the page alone.
- **Cost of a wrong call:**
  - *False positive:* editor hours spent on a page that was fine or would have recovered on its own.
  - *False negative:* a page that keeps losing traffic because it never reached the top of the queue. This is the costlier miss when the page still has real demand.
- **Why data or ML helps:** the signals that matter (impressions, clicks, position, age, freshness, engagement) are many and tangled, and the simplest-looking rule (search volume) already failed. A learned ranking can weigh them together, and I can test whether it beats a plain rule baseline using precision@50.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter CSV: how big the review backlog is, how weakly search volume tracks real traffic, and how the naive "biggest keywords first" rule does against random picking.

In [3]:
import os
import pandas as pd

# Load the starter CSV: local repo first, otherwise straight from GitHub (works in Colab)
LOCAL = "data/raw/content_refresh_anonymized.csv"
REMOTE = "https://raw.githubusercontent.com/YanoLeechiu/FlyRankAIProjects/main/" + LOCAL
path = next((os.path.join(up, LOCAL) for up in ["", "..", "../.."] if os.path.exists(os.path.join(up, LOCAL))), REMOTE)
df = pd.read_csv(path)
print(f"{len(df):,} pages from {df['client_id'].nunique()} clients\n")

# Number 1: how big is the review problem?
down_share = (df["trend_direction"] == "down").mean()
visible_down = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 500)
print(f"1) {down_share:.0%} of pages are trending down; {visible_down.sum():,} of those still get 500+ impressions.")
print(f"   With ~50 reviews a week, that is ~{visible_down.sum() / 50:.0f} weeks of work. Order matters.\n")

# Number 2: search volume barely tracks real traffic (my Week 1 finding)
rho = df[["search_volume", "impressions_90d", "clicks_90d"]].corr(method="spearman")["search_volume"]
print(f"2) Spearman correlation of search volume with impressions: {rho['impressions_90d']:+.2f}, with clicks: {rho['clicks_90d']:+.2f}\n")

# Number 3: the naive rule 'top 50 by search volume' vs. random picking
hits = visible_down[df.nlargest(50, "search_volume").index].sum()
print(f"3) Top 50 pages by search volume: {hits}/50 = {hits / 50:.0%} are visible decliners,")
print(f"   vs {visible_down.mean():.0%} if you picked pages at random. The naive rule is no better than chance.")


30,000 pages from 32 clients

1) 54% of pages are trending down; 9,961 of those still get 500+ impressions.
   With ~50 reviews a week, that is ~199 weeks of work. Order matters.

2) Spearman correlation of search volume with impressions: -0.03, with clicks: -0.07

3) Top 50 pages by search volume: 14/50 = 28% are visible decliners,
   vs 33% if you picked pages at random. The naive rule is no better than chance.


## 4. Careful words: what I can and can't claim

**What my work will be able to say:**
- *Observed:* which pages showed a measured drop in impressions, and which signals (position, age, freshness, engagement, clicks) tend to go along with that drop in this data.
- *Directional:* whether a learned ranking puts more truly declining pages in the top 50 than a simple rule does, measured with precision@50 on clients the model never saw.
- *Decision-support:* a ranked list of pages a person should **review first**, with reason codes so the reviewer can check why each page is there.

**What it will never say:**
- That refreshing a page **causes** it to recover. This data has no experiment, so I can only rank candidates, not promise results.
- That I found a **Google ranking factor** or can predict Google. These are observed patterns in one anonymized dataset.
- That a page flagged "declining" is guaranteed to keep declining. Some drops are seasonality, consolidation (a sibling page took the traffic), or low-volume noise.

**An honest limit of my numbers above:** "trending down" in the starter data compares the last 30 days of impressions with the 30 days before, and a page counts as down when it lost more than 20% (the cell below checks this). That is a label about the *current* window, not a future outcome. My capstone should move to a forward-looking label (features from the past 90 days, then a decline in the *next* 30 days) using the full warehouse data.

In [4]:
# Check: 'down' means last-30d impressions fell more than 20% vs the previous 30 days
print(df.groupby("trend_direction")["trend_pct"].agg(["count", "min", "max"]).round(1))


                 count    min      max
trend_direction                       
down             16262 -100.0    -20.0
flat                 0    NaN      NaN
new                  0    NaN      NaN
stable            5962  -20.0     20.0
up                4388   20.0  44900.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.